# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [46]:
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/Ahmedali3ff/Flyrank-internship-.git"
REPO_DIR = "/content/Flyrank-internship-"

# Clone repo if it does not exist
if not os.path.exists(REPO_DIR):
    subprocess.run(
            ["git", "clone", REPO_URL, REPO_DIR],
                    check=True
                        )
else:
                            print("Repository already exists.")

                            # Locate dataset
                            DATA_PATH = os.path.join(
                                REPO_DIR,
                                    "data",
                                        "raw",
                                            "content_refresh_anonymized.csv"
                                            )

                                            # Verify dataset exists
if not os.path.exists(DATA_PATH):
      raise FileNotFoundError(
                                                        f"Dataset not found at: {DATA_PATH}"
                                                            )

                                                            # Load dataset
df = pd.read_csv(DATA_PATH)
print("Repository:", REPO_DIR)
print("Dataset:", DATA_PATH)
print("Dataset loaded successfully.")
print("Shape:", df.shape)

Repository already exists.
Repository: /content/Flyrank-internship-
Dataset: /content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv
Dataset loaded successfully.
Shape: (30000, 44)


In [47]:
TARGET = "trend_direction"

df["is_declining"] = (
    df[TARGET].astype(str).str.lower() == "down"
    ).astype(int)

print("Declining rate:", df["is_declining"].mean())
print("\nTarget distribution:")
print(df["is_declining"].value_counts())

Declining rate: 0.5420666666666667

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64


In [48]:
# Signal #1: average search position

print("Average position by trend:")
display(
    df.groupby("is_declining")["avg_position"]
          .agg(["count", "mean", "median"])
          )

print("\nMedian position by trend:")
display(
              df.groupby("is_declining")["avg_position"]
                    .median()
                    )

Average position by trend:


,count,mean,median
is_declining,,,
0,13738,16.823060,10.05
1,16262,15.936305,11.30



Median position by trend:


,avg_position
is_declining,
0,10.05
1,11.30


In [49]:
# Signal #2: recent impressions

signal_cols = [
    "impressions_last_30d",
        "impressions_prev_30d",
        ]

print("Impression signals by trend:")

display(
            df.groupby("is_declining")[signal_cols]
                  .agg(["mean", "median"])
                  )

Impression signals by trend:


impressions_last_30d        impressions_prev_30d       
                             mean median                 mean median
is_declining                                                        
0                     2006.126656  156.0          1753.083928  103.5
1                      941.556635  128.0          1808.417661  313.0

In [50]:
# Signal #3: recent sessions

signal_cols = [
    "sessions_last_30d",
        "sessions_prev_30d",
        ]

print("Session signals by trend:")

display(
            df.groupby("is_declining")[signal_cols]
                  .agg(["mean", "median"])
                  )

Session signals by trend:


sessions_last_30d        sessions_prev_30d       
                          mean median              mean median
is_declining                                                  
0                    16.825666    3.0         11.333673    2.0
1                    11.823699    2.0          9.395400    2.0

In [51]:
# Compare simple decline-related flags

df["impression_decline_flag"] = (
    df["impressions_last_30d"] <
        df["impressions_prev_30d"]
        )

df["session_decline_flag"] = (
            df["sessions_last_30d"] <
                df["sessions_prev_30d"]
                )

flag_summary = pd.DataFrame({
                    "Impression decline flag": [
                            df.loc[df["impression_decline_flag"], "is_declining"].mean(),
                                    df.loc[~df["impression_decline_flag"], "is_declining"].mean()
                                        ],
                                            "Session decline flag": [
                                                    df.loc[df["session_decline_flag"], "is_declining"].mean(),
                                                            df.loc[~df["session_decline_flag"], "is_declining"].mean()
                                                                ]
                                                                }, index=["Flag = True", "Flag = False"])
display(flag_summary)

,Impression decline flag,Session decline flag
Flag = True,0.824812,0.623097
Flag = False,0.000000,0.501649


## Practical meaning

The audit identifies historical signals that are associated with the observed declining label.

These relationships are descriptive and directional. They should be interpreted as review-prioritization signals rather than causal drivers.

Search-position, impression, and session patterns can help prioritize content for further investigation, but the audit does not establish that changing any individual factor will cause performance to improve.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [52]:
# ============================================================
# 1. Build the feature vector
# ============================================================

import pandas as pd
import numpy as np

TARGET = "trend_direction"

CONTEXT_FIELDS = [
    "content_id",
        "client_id",
        ]

EXCLUDED_FIELDS = [
            "trend_direction",
                "trend_pct",
                ]

                # All remaining columns are candidate predictive features.
FEATURE_FIELDS = [
                    col for col in df.columns
                        if col not in CONTEXT_FIELDS + EXCLUDED_FIELDS
                        ]

                        # Separate numerical and categorical features
NUMERIC_FEATURES = [
                            col for col in FEATURE_FIELDS
                                if pd.api.types.is_numeric_dtype(df[col])
                                ]

CATEGORICAL_FEATURES = [
                                    col for col in FEATURE_FIELDS
                                        if not pd.api.types.is_numeric_dtype(df[col])
                                        ]

print("Total features:", len(FEATURE_FIELDS))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))

print("\nNumeric features:")
print(NUMERIC_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

Total features: 43
Numeric features: 32
Categorical features: 11

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'is_declining', 'impression_decline_flag', 'session_decline_flag']

Categorical features:
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [53]:
# Verify feature vector
X_raw = df[FEATURE_FIELDS].copy()

print("Feature matrix shape:", X_raw.shape)

print("\nMissing values:")
display(
    X_raw.isna()
             .sum()
                      .sort_values(ascending=False)
                               .to_frame("missing_count")
                               )

Feature matrix shape: (30000, 43)

Missing values:


,missing_count
provider_used,21438
word_count,7699
char_count,7699
char_count_tier,7699
word_count_tier,7699
model_used,5733
competition_level,2610
cpc,2468
search_volume,2468
competition,2468


## 2. Feature notes

The feature vector contains historical content, search-demand, engagement, freshness, and traffic-related signals.

### Numeric features

Numeric variables are retained in their observed numeric form. Missing numeric values are handled during preprocessing using median imputation, calculated from the training data only.

### Categorical features

Categorical variables such as content type, intent, provider/model, age tier, freshness tier, and other tier fields are handled using categorical encoding during modeling. Missing categorical values are represented explicitly before encoding.

### Availability

Features are intended to represent information available from the historical observation record before the prediction/review decision. Outcome-derived fields are excluded.

The model does not use `trend_direction` or `trend_pct` as predictors because these fields directly encode the outcome being predicted.

In [54]:
# ============================================================
# Feature notes: dtype, missingness, and observed coverage
# ============================================================

feature_notes = pd.DataFrame({
    "feature": FEATURE_FIELDS,
        "dtype": [str(df[c].dtype) for c in FEATURE_FIELDS],
            "missing_count": [df[c].isna().sum() for c in FEATURE_FIELDS],
                "missing_pct": [
                        df[c].isna().mean() * 100
                                for c in FEATURE_FIELDS
                                    ],
                                        "unique_values": [
                                                df[c].nunique(dropna=True)
                                                        for c in FEATURE_FIELDS
                                                            ],
                                                            })
feature_notes["type"] = np.where(
                                                                feature_notes["feature"].isin(NUMERIC_FEATURES),
                                                                    "numeric",
                                                                        "categorical"
                                                                        )

display(feature_notes)

,feature,dtype,missing_count,missing_pct,unique_values,type
0,search_volume,float64,2468,8.226667,41,numeric
1,competition,float64,2468,8.226667,101,numeric
2,competition_level,object,2610,8.700000,3,categorical
3,cpc,float64,2468,8.226667,915,numeric
4,content_type,object,0,0.000000,3,categorical
5,main_intent,object,2374,7.913333,4,categorical
6,word_count,float64,7699,25.663333,5476,numeric
7,char_count,float64,7699,25.663333,14839,numeric
8,provider_used,object,21438,71.460000,2,categorical
9,model_used,object,5733,19.110000,5,categorical


## 3. The leakage hunt

The leakage audit explicitly checks for:

1. Direct target-derived fields.
2. Fields whose names indicate trend/change/decline information.
3. Context identifiers accidentally entering the feature vector.
4. Features that may represent the outcome directly.
5. Rolling-window fields that must be interpreted as historical measurements rather than causal or future information.

The target is reconstructed from `trend_direction`, so `trend_direction` and `trend_pct` are excluded from the predictive feature vector.

In [55]:
# ============================================================
# Leakage Hunt 1: direct target-derived fields
# ============================================================

direct_leakage = [
    col for col in FEATURE_FIELDS
        if col in ["trend_direction", "trend_pct"]
        ]

print("Direct target-derived fields found in FEATURES:")
print(direct_leakage)

assert len(direct_leakage) == 0

Direct target-derived fields found in FEATURES:
[]


In [56]:
# ============================================================
# Leakage Hunt 2: suspicious feature names
# ============================================================

suspicious_terms = [
    "trend",
        "delta",
            "change",
                "growth",
                    "decline",
                    ]

suspicious_features = []

for col in FEATURE_FIELDS:
                        col_lower = col.lower()

if any(term in col_lower for term in suspicious_terms):
                                    suspicious_features.append(col)

                                    print("Suspicious feature names requiring review:")
                                    print(suspicious_features)

Suspicious feature names requiring review:
['session_decline_flag']


In [57]:
# ============================================================
# Leakage Hunt 3: context identifiers
# ============================================================

context_leakage = [
    col for col in FEATURE_FIELDS
        if col in CONTEXT_FIELDS
        ]

print("Context identifiers included as predictive features:")
print(context_leakage)

assert len(context_leakage) == 0

Context identifiers included as predictive features:
[]


In [58]:
# ============================================================
# Leakage Hunt 4: rolling-window fields
# ============================================================

window_features = [
    col for col in FEATURE_FIELDS
        if "30d" in col.lower() or "90d" in col.lower()
        ]

print("Rolling-window features:")
for col in window_features:
            print("-", col)

            print(
                "\nInterpretation: these fields describe historical "
                    "performance windows available in the observation record. "
                        "They are not treated as evidence of future performance."
                        )

Rolling-window features:
- impressions_90d

Interpretation: these fields describe historical performance windows available in the observation record. They are not treated as evidence of future performance.
- clicks_90d

Interpretation: these fields describe historical performance windows available in the observation record. They are not treated as evidence of future performance.
- pageviews_90d

Interpretation: these fields describe historical performance windows available in the observation record. They are not treated as evidence of future performance.
- sessions_90d

Interpretation: these fields describe historical performance windows available in the observation record. They are not treated as evidence of future performance.
- users_90d

Interpretation: these fields describe historical performance windows available in the observation record. They are not treated as evidence of future performance.
- engaged_sessions_90d

Interpretation: these fields describe historical performance w

## 4. What I excluded and why

| Field | Decision | Reason |
|---|---|---|
| `trend_direction` | Excluded | Directly defines the target label and would cause target leakage. |
| `trend_pct` | Excluded | Directly represents the measured trend outcome and would leak the target information. |
| `content_id` | Context only | An anonymized identifier used for traceability; it has no meaningful predictive interpretation. |
| `client_id` | Context only | Used for client-grouped validation; including it as a predictive feature could allow the model to memorize client-specific patterns. |

In [59]:
# ============================================================
# Final exclusion audit
# ============================================================

exclusion_table = pd.DataFrame({
    "field": [
            "trend_direction",
                    "trend_pct",
                            "content_id",
                                    "client_id",
                                        ],
                                            "status": [
                                                    "excluded",
                                                            "excluded",
                                                                    "context only",
                                                                            "context only",
                                                                                ],
                                                                                    "reason": [
                                                                                            "Directly defines the target label.",
                                                                                                    "Directly represents the measured trend outcome.",
                                                                                                            "Identifier only; not a meaningful predictive feature.",
                                                                                                                    "Used for grouped validation; excluded from prediction to avoid client memorization.",
                                                                                                                        ],
                                                                                                                        })
display(exclusion_table)

,field,status,reason
0,trend_direction,excluded,Directly defines the target label.
1,trend_pct,excluded,Directly represents the measured trend outcome.
2,content_id,context only,Identifier only; not a meaningful predictive f...
3,client_id,context only,Used for grouped validation; excluded from pre...


In [60]:
# ============================================================
# Final self-check
# ============================================================

assert TARGET not in FEATURE_FIELDS
assert "trend_pct" not in FEATURE_FIELDS
assert "content_id" not in FEATURE_FIELDS
assert "client_id" not in FEATURE_FIELDS

print("Leakage checks passed.")
print("Final feature count:", len(FEATURE_FIELDS))

Leakage checks passed.
Final feature count: 43


## Self-check

- [ ] Three measurable signals were audited.
- [ ] Results are based on observed historical data.
- [ ] No causal claim is made.
- [ ] Signals are treated as directional decision-support evidence.
- [ ] The notebook runs top to bottom without errors.
- [ ] No client names, private URLs, queries, or credentials are included.